In [4]:
import sys
import os
import pandas as pd

# Add scripts directory to path
project_root = os.path.abspath('../..')  # Go up to project root
scripts_path = os.path.join(project_root, 'scripts')
sys.path.insert(0, scripts_path)

# Now import
from utils import load_data  # Not "from scripts.utils"

# Load data
df = load_data()

Loading from /Users/carlottaharper/Desktop/sem 2/data structures and algorithms/project-chaggg/data/cleaned/chicago_crimes_cleaned.parquet...


In [5]:
df.head()

,id,case_number,date,block,iucr,primary_type,description,location_description,arrest,domestic,...,ward,community_area,fbi_code,year,updated_on,x_coordinate,y_coordinate,latitude,longitude,time
0,13246664,JG467702,2001-01-01,045XX N CENTRAL PARK AVE,0281,CRIMINAL SEXUAL ASSAULT,NON-AGGRAVATED,RESIDENCE,False,False,...,33.0,14.0,02,2001,2023-10-21T15:42:03.000,NaN,NaN,NaN,NaN,00:00:00
1,13714017,JH548973,2001-01-01,051XX W MEDILL AVE,1751,OFFENSE INVOLVING CHILDREN,CRIMINAL SEXUAL ABUSE BY FAMILY MEMBER,APARTMENT,False,True,...,31.0,19.0,17,2001,2025-01-11T15:40:58.000,NaN,NaN,NaN,NaN,00:00:00
2,13714063,JH548896,2001-01-01,051XX W MEDILL AVE,1751,OFFENSE INVOLVING CHILDREN,CRIMINAL SEXUAL ABUSE BY FAMILY MEMBER,APARTMENT,False,True,...,31.0,19.0,17,2001,2025-01-11T15:40:58.000,NaN,NaN,NaN,NaN,00:00:00
3,1310933,G001704,2001-01-01,0000X W WACKER DR,0810,THEFT,OVER $500,PARKING LOT/GARAGE(NON.RESID.),False,False,...,NaN,NaN,06,2001,2015-08-17T15:03:40.000,1176294.0,1902094.0,41.886704,-87.628054,00:00:00
4,1432149,G156662,2001-01-01,002XX E DELAWARE PL,2825,OTHER OFFENSE,HARASSMENT BY TELEPHONE,RESIDENCE,False,False,...,NaN,NaN,26,2001,2015-08-17T15:03:40.000,1178492.0,1906652.0,41.899161,-87.619844,00:00:00


In [7]:
import math

def haversine(lat1, lon1, lat2, lon2):
    """
    Calculate the great-circle distance between two points on Earth.
    
    Parameters:
        lat1, lon1: Latitude and longitude of point 1 (in decimal degrees)
        lat2, lon2: Latitude and longitude of point 2 (in decimal degrees)
    
    Returns:
        Distance in kilometers
    """

    # Convert degrees to radians
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])

    # Differences
    dlat = lat2 - lat1
    dlon = lon2 - lon1

    # Haversine formula
    a = math.sin(dlat / 2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))#atan treats edge cases better on sphere
    R = 6371  # Earth's radius in kilometers
    return R * c

In [9]:
def temporal_distance(month1, day1, hour1, month2, day2, hour2):
    """
    Calculate normalized temporal distance between two crimes.
    
    Parameters:
        month1, day1, hour1: Time components of crime 1
        month2, day2, hour2: Time components of crime 2
    
    Returns:
        Temporal distance score between 0 and 1
    """
    # Normalize each component to [0, 1]
    month_diff = abs(month1 - month2) / 12      # months range 1-12
    day_diff = abs(day1 - day2) / 31            # days range 1-31
    hour_diff = abs(hour1 - hour2) / 24         # hours range 0-23

    # Average the three normalized differences
    return (month_diff + day_diff + hour_diff) / 3

In [ ]:
def combined_distance(crime1, crime2, alpha=0.5, beta=0.5):
    """
    Combined spatial and temporal distance between two crimes.
    
    Parameters:
        crime1, crime2: dicts or rows with keys 'latitude', 'longitude', 'month', 'day', 'hour'
        alpha: weight for spatial distance
        beta: weight for temporal distance
    
    Returns:
        Combined distance score
    """
    spatial = haversine(crime1['latitude'], crime1['longitude'], 
                        crime2['latitude'], crime2['longitude'])
    temporal = temporal_distance(crime1['month'], crime1['day'], crime1['hour'],
                                 crime2['month'], crime2['day'], crime2['hour'])
    
    return alpha * spatial + beta * temporal